In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder


In [13]:
df = pd.read_csv('../Data/cw_22_23_24.csv')
df.head(5)

,adm_type,shift_from,ssc,yr_nae,m_no,mrn,pt_name,sex,disease,D.O.A,D.O.D,status,consultant,L.O.S
0,Shift From,ER,No,1,1,21845698,Hara Bibi,F,STEMI,1-Jan-22,1-Jan-22,Discharge,Imran Khan,0
1,Shift From,ER,No,2,2,22000071,Taj Rehman,M,ADHF,1-Jan-22,5-Jan-22,Discharge,Malik Faisal,4
2,Shift From,ER,No,3,3,21838760,Bakhtawar Shah,M,ihd,1-Jan-22,10-Jan-22,Discharge,Asif Iqbal,9
3,Shift From,ER,No,4,4,22000251,Arasal Jan Bibi,F,NaN,1-Jan-22,7-Jan-22,Discharge,Sher Bahadar,6
4,Shift From,Neu,No,5,5,21825110,Khad Mewa,F,NaN,1-Jan-22,2-Jan-22,Discharge,Tariq Nawaz,1


In [14]:
missing_data = df.isnull().sum()
print(missing_data)


adm_type         0
shift_from       0
ssc              0
yr_nae           0
m_no             0
mrn              0
pt_name          0
sex              0
disease       2092
D.O.A            0
D.O.D            0
status           0
consultant       0
L.O.S            0
dtype: int64


In [15]:
missing_di_data = df['disease'].isnull().sum()
print(missing_di_data)

2092


In [16]:
total_diseases = df['disease'].count()
total_diseases

np.int64(7481)

In [17]:
perc_of_missing_data = missing_di_data/total_diseases
perc_of_missing_data


np.float64(0.2796417591231119)

In [18]:
df['disease'] = df['disease'].fillna('Missing')

contingency = pd.crosstab(df['disease'], df['status'])

chi2, p, dof, expected = chi2_contingency(contingency)

print("Chi-square statistic:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p)



Chi-square statistic: 4266.103207150643
Degrees of freedom: 3488
p-value: 1.5278190290826269e-18


In [19]:
count_dupe = df.duplicated().sum()
count_dupe

np.int64(0)

In [20]:
df.dtypes


adm_type      object
shift_from    object
ssc           object
yr_nae         int64
m_no           int64
mrn           object
pt_name       object
sex           object
disease       object
D.O.A         object
D.O.D         object
status        object
consultant    object
L.O.S          int64
dtype: object

In [42]:
# Convert to datetime
df['D.O.A'] = pd.to_datetime(df['D.O.A'], format='%d-%b-%y', errors='coerce')
df['D.O.D'] = pd.to_datetime(df['D.O.D'], format='%d-%b-%y', errors='coerce')

# Extract year/month
df['DOA_year'] = df['D.O.A'].dt.year
df['DOA_month'] = df['D.O.A'].dt.month
df['DOD_year'] = df['D.O.D'].dt.year
df['DOD_month'] = df['D.O.D'].dt.month

# One-hot encode categorical columns
categorical_columns = ['shift_from', 'disease', 'status', 'sex']

# Handle missing values
df[categorical_columns] = df[categorical_columns].fillna("Missing")

encoder = OneHotEncoder(sparse_output=False)
one_hot_encoded = encoder.fit_transform(df[categorical_columns])

one_hot_df = pd.DataFrame(
    one_hot_encoded,
    columns=encoder.get_feature_names_out(categorical_columns),
    index=df.index  # keep same index to avoid misalignment
)

# Combine with original df and drop original categorical columns
df_encoded = pd.concat([df, one_hot_df], axis=1).drop(categorical_columns, axis=1)

print(f"Encoded data:\n{df_encoded.head()}")


Encoded data:
     adm_type ssc  yr_nae  m_no       mrn          pt_name      D.O.A  \
0  Shift From  No       1     1  21845698        Hara Bibi 2022-01-01   
1  Shift From  No       2     2  22000071       Taj Rehman 2022-01-01   
2  Shift From  No       3     3  21838760   Bakhtawar Shah 2022-01-01   
3  Shift From  No       4     4  22000251  Arasal Jan Bibi 2022-01-01   
4  Shift From  No       5     5  21825110       Khad Mewa  2022-01-01   

       D.O.D    consultant  L.O.S  ...  disease_wallen type "B"  \
0 2022-01-01    Imran Khan      0  ...                      0.0   
1 2022-01-05  Malik Faisal      4  ...                      0.0   
2 2022-01-10    Asif Iqbal      9  ...                      0.0   
3 2022-01-07  Sher Bahadar      6  ...                      0.0   
4 2022-01-02   Tariq Nawaz      1  ...                      0.0   

   disease_wellon syndrom  disease_wmi  status_DOW  status_Discharge  \
0                     0.0          0.0         0.0               1.0   


In [48]:
df_encoded.dtypes
df_encoded.head(1)

,adm_type,ssc,yr_nae,m_no,mrn,pt_name,D.O.A,D.O.D,consultant,L.O.S,...,"disease_wallen type ""B""",disease_wellon syndrom,disease_wmi,status_DOW,status_Discharge,status_Expire,status_LAMA,status_Shifted,sex_F,sex_M
0,Shift From,No,1,1,21845698,Hara Bibi,2022-01-01,2022-01-01,Imran Khan,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [51]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report


In [57]:
def splitdataset(balance_data):
    X = balance_data[['disease_wallen type "B"', 'disease_wellon syndrom', 'disease_wmi', 'sex_F', 'sex_M']]
    Y = balance_data[['status_DOW', 'status_Discharge', 'status_Expire', 'status_LAMA', 'status_Shifted']]

    X_train, X_test, y_train, y_test = train_test_split(
        X, Y, test_size=0.3, random_state=100)

    return X, Y, X_train, X_test, y_train, y_test


def cal_accuracy(y_test, y_pred):
    print("Accuracy : ", accuracy_score(y_test, y_pred) * 100)
    print("Report : \n", classification_report(y_test, y_pred))

X, Y, X_train, X_test, y_train, y_test = splitdataset(df_encoded)

clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
cal_accuracy(y_test, y_pred)

Accuracy :  89.55431754874652
Report : 
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        15
           1       0.90      1.00      0.94      2572
           2       0.00      0.00      0.00       168
           3       0.00      0.00      0.00        57
           4       0.00      0.00      0.00        60

   micro avg       0.90      0.90      0.90      2872
   macro avg       0.18      0.20      0.19      2872
weighted avg       0.80      0.90      0.85      2872
 samples avg       0.90      0.90      0.90      2872



C:\Users\jacki\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
